##### Benefits of Multi-Agent Architecture
1. Context Isolation
2. Specialization
3. Parallelism
4. Fault Isolation

* Rule of thumb: Use multi-agent when a task requires more than 3 distinct knowledge domains or more than 50K tokens of context. Below that threshold, a single well-prompted agent is simpler, cheaper, and often faster.

##### Orchestration Patterns
1. Sequential Pipeline
2. Parallel Fan-Out / Fan-In
3. Hierarchical Delegation. A top-level coordinator delegates to team leads, who further delegate to workers. This creates a tree of management -- mirroring how large engineering teams actually operate.
4. Conditional Routing:A router agent inspects each incoming task and directs it to the most appropriate specialist. Different inputs take different paths through the system.

* The "agent-sized chunk" heuristic: Each subtask should be completable within 20-30 tool calls. If a subtask requires more, it should be decomposed further. If it requires fewer than 5, it's probably too granular and should be merged with another.

##### State Management and Communication
###### Shared State Patterns
1. State File on Disk (JSON): The simplest pattern. A single JSON file on disk that all agents read from and write to. The coordinator manages writes to prevent conflicts.
2. Event-Driven (WebSocket / SSE): Agents publish events as they make progress. Other agents (and the UI) subscribe to these events. State is reconstructed from the event stream.
3. Database-Backed: Agents read from and write to a shared database. Transactions ensure consistency. Queries enable complex state lookups.
4. Inter-Agent Communication
4.1. Hub-and-Spoke (Through the Coordinator): Agents never communicate directly. All messages flow through the coordinator. Agent A sends results to the coordinator, which decides what to pass to Agent B.
4.2. Via Shared Files: Agents read from and write to shared file locations. Agent A writes analysis-results.json, Agent B reads it. The file system is the communication channel.
4.3. Via Database: Agents write their results to database tables. Other agents query those tables for input. This is hub-and-spoke with the database as the hub.

** Structured Inter-Agent Message **
{
  "source_agent": "security-reviewer",
  "timestamp": "2025-01-15T14:30:00Z",
  "confidence": 0.92,
  "findings": [
    {
      "severity": "high",
      "file": "src/auth/login.py",
      "line": 47,
      "issue": "SQL injection via unsanitized user input",
      "recommendation": "Use parameterized query"
    }
  ],
  "summary": "1 high-severity, 2 medium-severity issues found. No critical vulnerabilities."
}

##### The Jumpstarter Pattern
{
  "current_phase": "api",
  "completed_phases": ["scaffold", "database"],
  "remaining_phases": ["api", "frontend", "tests", "qa"],
  "iteration": 3,
  "max_iterations": 15,
  "phase_results": {
    "scaffold": { "status": "complete", "files_created": 12 },
    "database": { "status": "complete", "tables": 8 }
  }
}

* Agents: Scaffold Architect, Data Architect, API Architect, Frontend Architect, QA Validator

* The Stop Hook Pattern: The key innovation is the **stop hook**. After each Claude Code session completes (or hits its context limit), the stop hook checks **.jumpstarter-state.json**. If there are remaining phases, it automatically restarts Claude with a fresh context window, pointed at the next phase. This allows the system to generate far more code than fits in a single context window.

##### Failure Handling and Recovery
###### Fallback Agents
If a specialist agent fails, try a generalist. A dedicated security reviewer might fail on an unusual codebase, but a **general-purpose Opus agent** can often produce a reasonable (if less targeted) analysis

# Pseudocode: fallback agent strategy
try:
    result = run_specialist_agent("security-reviewer", task)
except AgentFailure:
    # Specialist failed -- fall back to generalist
    result = run_generalist_agent("opus-general", task)
    result.metadata["fallback"] = True


